In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Any-content frame — text, image, PDF across 4 vendors
# Matches diagrams: slide4a_any_content + image_delivery_base64
# ─────────────────────────────────────────────────────────────
import base64
import os
from dotenv import load_dotenv

load_dotenv()
print("Keys loaded from .env")
print()

In [2]:
# ── Guard: sample files must exist ───────────────────────────
for fname in ["sample_claim.txt", "sample_invoice.png", "sample_invoice.pdf"]:
    if not os.path.exists(fname):
        raise FileNotFoundError(
            f"Missing: {fname}\n"
            f"Place your sample files in the same folder as this script."
        )
print("Sample files found ✓")
print()

# ── Load files ────────────────────────────────────────────────
TEXT = open("sample_claim.txt", "r").read()
IMG  = open("sample_invoice.png", "rb").read()
PDF  = open("sample_invoice.pdf", "rb").read()

def b64(raw):
    """Bytes → base64 string (what actually rides inside the JSON frame)."""
    return base64.b64encode(raw).decode("utf-8")

# A tiny question so the reply (and the bill) stays short
ASK = "What is the TOTAL amount on this document? Reply with just the number."

# What 'going out' looks like, per modality (vendor-independent part).
OUTGOING = {
    "text":  ("inline string", len(TEXT.encode()), None),
    "image": ("base64 inline", len(IMG), len(b64(IMG))),
    "pdf":   ("base64 inline", len(PDF), len(b64(PDF))),
}

for m, (enc, raw, b) in OUTGOING.items():
    extra = f" -> {b:,} b64 chars (+{(b/raw-1)*100:.0f}%)" if b else ""
    print(f"{m:6s} {enc:14s} {raw:>8,} bytes{extra}")

Sample files found ✓

text   inline string       226 bytes
image  base64 inline    21,453 bytes -> 28,604 b64 chars (+33%)
pdf    base64 inline     1,379 bytes -> 1,840 b64 chars (+33%)


In [ ]:
# ── Initialise clients once ───────────────────────────────────
from openai    import OpenAI
from anthropic import Anthropic
from google    import genai
from groq      import Groq

openai_client    = OpenAI()
anthropic_client = Anthropic()
google_client    = genai.Client()
groq_client      = Groq()
print("All clients initialised ✓")
print()


# ════════════════════════════════════════════════════════════
print("=" * 65)
print("DEMO 1 — payload packaging: what goes over the wire")
print("=" * 65)
print()
# ════════════════════════════════════════════════════════════

# ── Show raw vs base64 sizes ──────────────────────────────────
print("WHY +33%?  base64 encodes 3 raw bytes as 4 ASCII characters.")
print("The HTTP gateway sees the base64 size — this is what Guard 1 checks.")
print()
print(f"  {'MODALITY':8s}  {'RAW BYTES':>12s}  {'BASE64 CHARS':>14s}  INFLATION")
print("  " + "-" * 52)
print(f"  {'text':8s}  {len(TEXT.encode()):>12,}  {'(inline string)':>14s}  —")
print(f"  {'image':8s}  {len(IMG):>12,}  {len(b64(IMG)):>14,}  "
      f"+{(len(b64(IMG))/len(IMG)-1)*100:.0f}%")
print(f"  {'pdf':8s}  {len(PDF):>12,}  {len(b64(PDF)):>14,}  "
      f"+{(len(b64(PDF))/len(PDF)-1)*100:.0f}%")
print()
print("A large PDF base64-encoded can trigger 413 at Guard 1 even if it")
print("would fit inside the model's context window — the gateway checks bytes,")
print("not tokens. Two separate size checks, two separate guards.")
print()

# ── Show how each vendor wraps each modality ──────────────────

PACKAGING = {
    "Anthropic": {"text": "content string", "image": "image/base64 block",  "pdf": "document/base64 block"},
    "OpenAI":    {"text": "content string", "image": "image_url data: URI", "pdf": "file/file_data data: URI"},
    "Google":    {"text": "contents string","image": "Part.from_bytes",     "pdf": "Part.from_bytes"},
    "Groq":      {"text": "content string", "image": "image_url data: URI", "pdf": "- not supported -"},
}
MODALITIES = ["text", "image", "pdf"]

print(f"  {'':12s}" + "".join(f"  {v:<28s}" for v in PACKAGING))
print("  " + "-" * 100)
for m in MODALITIES:
    row = f"  {m:12s}"
    for v in PACKAGING:
        row += f"  {PACKAGING[v][m]:<28s}"
    print(row)
print()
print("Same content, four different envelope formats — this is why raw HTTP")
print("differs per vendor even though the Python SDK hides it from you.")
print()

# ════════════════════════════════════════════════════════════
print("=" * 65)
print("DEMO 2 — pre-flight token estimates (tiktoken, before sending)")
print("=" * 65)
print()
# ════════════════════════════════════════════════════════════

import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # GPT-4o tokenizer

text_tok  = len(enc.encode(TEXT))
# Images and PDFs are not text — we estimate from base64 length as a proxy.
# Actual image tokens depend on vendor tiling (see Demo 3 results).
image_est = len(b64(IMG)) // 4
pdf_est   = len(b64(PDF)) // 4

print("Connecting this demo back to the tokenization session:")
print(f"  text  : {text_tok:>8,} tokens  (exact — text runs through tokenizer)")
print(f"  image : {image_est:>8,} tokens  (rough estimate — vendor tiling math varies)")
print(f"  pdf   : {pdf_est:>8,} tokens  (rough estimate — vendor tiling math varies)")
print()
print("Images are NOT free — each vendor converts pixels into token patches.")
print("A 1024×1024 image can cost 500–1500 tokens depending on the vendor.")
print("This is why image-heavy apps burn through context window fast.")
print()


# ════════════════════════════════════════════════════════════
print("=" * 65)
print("DEMO 3 — send all three modalities to all four vendors")
print("=" * 65)
print()
# ════════════════════════════════════════════════════════════

In [4]:
# ── Helper result builders ────────────────────────────────────
def ok(in_tok, out_tok, reply):
    return {
        "status": "200",
        "in": in_tok,
        "out": out_tok,
        "reply": (reply or "").strip()[:40],
    }

def err(status):
    return {"status": status, "in": None, "out": None, "reply": ""}

def na(note):
    return {"status": "n/a", "in": None, "out": None, "reply": note}

def status_of(error):
    """Extract HTTP status code from any vendor SDK error."""
    code = getattr(error, "status_code", None) or getattr(error, "code", None)
    if code:
        return str(code)
    grpc_map = {
        "INVALID_ARGUMENT":   "400",
        "RESOURCE_EXHAUSTED": "429",
        "INTERNAL":           "500",
        "UNAVAILABLE":        "503",
    }
    for grpc_name, http_code in grpc_map.items():
        if grpc_name in str(error).upper():
            return http_code
    for guess in ("400", "413", "415", "422", "429", "500", "503"):
        if guess in str(error):
            return guess
    return "error"

In [5]:
# ── Send functions ────────────────────────────────────────────

def send_anthropic(modality):
    if modality == "text":
        content = TEXT
    elif modality == "image":
        content = [
            {"type": "text", "text": ASK},
            {"type": "image", "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": b64(IMG),
            }},
        ]
    else:  # pdf
        content = [
            {"type": "text", "text": ASK},
            {"type": "document", "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": b64(PDF),
            }},
        ]
    try:
        r = anthropic_client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=10,
            messages=[{"role": "user", "content": content}],
        )
        reply = r.content[0].text if r.content else ""
        return ok(r.usage.input_tokens, r.usage.output_tokens, reply)
    except Exception as e:
        return err(status_of(e))



In [6]:
def send_openai(modality):
    if modality == "text":
        content = TEXT
    elif modality == "image":
        content = [
            {"type": "text", "text": ASK},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{b64(IMG)}"}},
        ]
    else:  # pdf
        # NOTE: file_data in chat completions is a recent OpenAI addition.
        # If you get 400 or 422, your account tier may require the
        # Files API upload approach (client.files.create) instead.
        content = [
            {"type": "text", "text": ASK},
            {"type": "file", "file": {
                "filename": "invoice.pdf",
                "file_data": f"data:application/pdf;base64,{b64(PDF)}",
            }},
        ]
    try:
        r = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=10,
            messages=[{"role": "user", "content": content}],
        )
        u = r.usage
        return ok(u.prompt_tokens, u.completion_tokens,
                  r.choices[0].message.content)
    except Exception as e:
        return err(status_of(e))

In [7]:
def send_google(modality):
    from google.genai import types
    if modality == "text":
        contents = TEXT
    elif modality == "image":
        contents = [ASK, types.Part.from_bytes(data=IMG, mime_type="image/png")]
    else:  # pdf — Gemini accepts raw bytes + mime_type, same as images
        contents = [ASK, types.Part.from_bytes(data=PDF,
                    mime_type="application/pdf")]
    try:
        r = google_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=contents,
        )
        u = r.usage_metadata
        return ok(u.prompt_token_count, u.candidates_token_count, r.text or "")
    except Exception as e:
        return err(status_of(e))

In [8]:
def send_groq(modality):
    if modality == "text":
        model   = "llama-3.3-70b-versatile"
        content = TEXT
    elif modality == "image":
        # Vision and text live on DIFFERENT Groq models
        model   = "meta-llama/llama-4-scout-17b-16e-instruct"
        content = [
            {"type": "text", "text": ASK},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{b64(IMG)}"}},
        ]
    else:  # pdf — Groq has no document input type in their API spec
        print(f"    SKIP — Groq has no PDF/document input type")
        return na("Groq has no PDF/document block")
    try:
        r = groq_client.chat.completions.create(
            model=model,
            max_tokens=10,
            messages=[{"role": "user", "content": content}],
        )
        u = r.usage
        return ok(u.prompt_tokens, u.completion_tokens,
                  r.choices[0].message.content)
    except Exception as e:
        return err(status_of(e))

In [10]:
VENDORS = {
    "Anthropic": send_anthropic,
    "OpenAI":    send_openai,
    "Google":    send_google,
    "Groq":      send_groq,
}

In [ ]:
# ── Run the sweep ─────────────────────────────────────────────
grid = {m: {} for m in MODALITIES}

for modality in MODALITIES:
    enc_label, raw_bytes, b64_chars = {
        "text":  ("inline string",  len(TEXT.encode()), None),
        "image": ("base64 inline",  len(IMG),           len(b64(IMG))),
        "pdf":   ("base64 inline",  len(PDF),           len(b64(PDF))),
    }[modality]

    size_str = f"{raw_bytes:,} B"
    if b64_chars:
        size_str += f" → {b64_chars:,} b64 chars (+33%)"

    print("─" * 65)
    print(f"MODALITY: {modality.upper():<6s}  encoding: {enc_label}  size: {size_str}")
    print("─" * 65)

    for vendor, send_fn in VENDORS.items():
        wrap = PACKAGING[vendor][modality]
        print(f"  OUT → {vendor:10s}  [{wrap}]")
        res  = send_fn(modality)
        grid[modality][vendor] = res

        if res["status"] == "200":
            print(f"  IN  ← 200   "
                  f"in={res['in']:>6,} tok  "
                  f"out={res['out']:>3} tok  "
                  f"reply={res['reply']!r}")
        elif res["status"] == "n/a":
            print(f"  IN  ← n/a   ({res['reply']})")
        else:
            print(f"  IN  ← {res['status']}   "
                  f"(rejected — this modality/vendor combo failed)")
        print()

print()


# ════════════════════════════════════════════════════════════
print("=" * 65)
print("DEMO 4 — summary grids")
print("=" * 65)
print()
# ════════════════════════════════════════════════════════════

In [ ]:
# ── Status grid ───────────────────────────────────────────────
print("STATUS GRID — what each vendor did with each modality")
print()
print(f"  {'MODALITY':10s}" + "".join(f"  {v:>12s}" for v in VENDORS))
print("  " + "-" * 58)
for m in MODALITIES:
    row = f"  {m:10s}"
    for v in VENDORS:
        row += f"  {grid[m][v]['status']:>12s}"
    print(row)
print("  " + "-" * 58)
print("  200 = accepted   400/413/415/422 = rejected   n/a = not supported")
print("  429 = rate limit   413 = payload too large for gateway")
print()

In [ ]:
# ── Token grid ────────────────────────────────────────────────
print("INPUT TOKEN GRID — same content, counted differently per vendor")
print()
print(f"  {'MODALITY':10s}" + "".join(f"  {v:>12s}" for v in VENDORS))
print("  " + "-" * 58)
for m in MODALITIES:
    row = f"  {m:10s}"
    for v in VENDORS:
        t = grid[m][v]["in"]
        row += f"  {f'{t:,}' if t is not None else '-':>12s}"
    print(row)
print("  " + "-" * 58)
print()
print("KEY OBSERVATIONS:")
print()
print("  1. IMAGE TOKENS — each vendor tiles pixels differently.")
print("     The same image costs a different number of tokens per vendor.")
print("     This is vendor-specific tiling math, not a bug.")
print()
print("  2. PDF TOKENS — PDFs are parsed to text+layout then tokenised.")
print("     A scanned PDF (image-only) costs image tokens, not text tokens.")
print()
print("  3. BASE64 INFLATION — the +33% byte overhead is a Guard 1 concern,")
print("     not a token concern. Tokens count the decoded content,")
print("     not the base64 wrapper around it.")
print()
print("  4. GROQ PDF — Groq simply does not have a document input type.")
print("     In production you would pre-extract text with pdfplumber or")
print("     pypdf and send it as a text block instead.")